In [3]:
import torch
import torch.nn as nn
import numpy as np

# 1. Dataset Preprocessing Matrix Mapping
with open('/content/t8.shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Encode entire document into integer representations
data = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)

def get_batches(data, batch_size, seq_length):
    inputs = torch.zeros(batch_size, seq_length, dtype=torch.long)
    targets = torch.zeros(batch_size, seq_length, dtype=torch.long)

    # Random selection windows across text string bounds
    for i in range(batch_size):
        start_idx = np.random.randint(0, len(data) - seq_length - 1)
        inputs[i] = data[start_idx : start_idx + seq_length]
        targets[i] = data[start_idx + 1 : start_idx + seq_length + 1]
    return inputs, targets

# 2. Gated Recurrent Network Architecture
class TextGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(TextGRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.gru(x, hidden)
        out = self.fc(out)
        return out, hidden

# Model Configurations
hidden_dim = 256
model = TextGRU(vocab_size, embed_dim=128, hidden_dim=hidden_dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

# 3. Model Training Sequence Routine
print("     INITIALIZING SEQUENTIAL GRU TRAINING    ")

batch_size = 64
seq_length = 100

for step in range(1200):
    inputs, targets = get_batches(data, batch_size, seq_length)
    inputs, targets = inputs.to(device), targets.to(device)

    outputs, _ = model(inputs)
    # Reshape vectors for cross-entropy evaluation metrics
    loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (step + 1) % 300 == 0:
        print(f"Step {step+1:04d}/1200 | Categorical Cross-Entropy Loss: {loss.item():.4f}")

# 4. Interactive Temperature-Controlled Inference Engine
def generate_text(model, start_str='THE ', gen_length=150, temperature=0.7):
    model.eval()
    hidden = None
    input_seq = torch.tensor([char_to_idx[ch] for ch in start_str], dtype=torch.long).unsqueeze(0).to(device)
    generated_output = start_str

    # Prime the hidden state matrices with our starting seed string
    for i in range(len(start_str) - 1):
        _, hidden = model(input_seq[:, i].unsqueeze(1), hidden)

    curr_char_tensor = input_seq[:, -1].unsqueeze(1)

    with torch.no_grad():
        for _ in range(gen_length):
            outputs, hidden = model(curr_char_tensor, hidden)

            # Apply scaling temperature to flatten or sharpen distributions
            logits = outputs.squeeze(1) / temperature
            probs = torch.softmax(logits, dim=-1)

            # Non-deterministic token selection via multinomial distributions
            next_idx = torch.multinomial(probs, num_samples=1).item()
            generated_output += idx_to_char[next_idx]

            curr_char_tensor = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    print(f"     INFERENCE SEED INPUT: '{start_str}' (TEMP: {temperature}")
    print(generated_output)
generate_text(model, start_str="KING: ", gen_length=200000, temperature=0.7)

     INITIALIZING SEQUENTIAL GRU TRAINING    
Step 0300/1200 | Categorical Cross-Entropy Loss: 1.5776
Step 0600/1200 | Categorical Cross-Entropy Loss: 1.3985
Step 0900/1200 | Categorical Cross-Entropy Loss: 1.3551
Step 1200/1200 | Categorical Cross-Entropy Loss: 1.3106
     INFERENCE SEED INPUT: 'KING: ' (TEMP: 0.7
KING: and you shall I must have dead, and then;
    Be a through the orcharg, we are the strong love?
  IMOGEN. I will rest that your dead,
    And I'll answer that he is one of from sleep for sure.
    There will I am whole uncle of the sleep of better,
    And sword of stoly, what do posinion.
  LAFENBANT. The sleeper, and play the soft since of love?'
    And then I for the means, and you throw doth like it.
    What shall I pray him even for me;
    But yet my soul and sorrow that return the child,
    And love or so much do.  
    Here something drinks and head would seen the report
    Which you will go seen to our thing in horse;
    The seeph.                        